# Module 05 — Neural Network Foundations
## 05-06: Backpropagation

**Objective:** Derive and implement backpropagation from scratch using the chain rule, verify
correctness with numerical gradient checking, and compare against PyTorch autograd.

**Prerequisites:** 05-02 (MLP architecture), 05-04 (loss functions), 01-03 (Calculus — gradient
descent mechanics).


## Part 0 — Setup & Prerequisites

Backpropagation is the engine behind neural-network training. Given a loss $\mathcal{L}$ and
any parameter $\theta$, backprop computes $\partial \mathcal{L}/\partial \theta$ exactly via
the **chain rule** applied layer-by-layer through the computational graph.

In this notebook we:
1. Review the scalar and vector chain rules with concrete examples.
2. Implement `linear_forward`, `relu_forward`, `mse_loss`, and their backward counterparts
   from scratch in NumPy.
3. Assemble them into a `ManualMLP` that trains entirely without PyTorch autograd.
4. Verify every gradient with a **numerical gradient check** (finite differences).
5. Compare our gradients and training trajectory against PyTorch autograd.
6. Measure how gradient norms change with network depth (vanishing gradient preview).

> **Concept ownership:** `PyTorch autograd` internals (computation graph, `grad_fn`, custom
> `Function`) are covered fully in **05-07**. Here we treat autograd as a black box for comparison.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler

import random
from typing import Callable
print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy:   {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA:    {torch.version.cuda}")
    print(f"GPU:     {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
# Dataset
N_SAMPLES    = 1_000     # synthetic regression samples
N_FEATURES   = 20        # input dimensionality
N_INFORMATIVE = 15       # features that actually drive the target

# Architecture
HIDDEN_SIZES = [64, 32]  # hidden layer widths
OUTPUT_SIZE  = 1         # regression output

# Training
BATCH_SIZE    = 64
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-2     # SGD learning rate (higher than Adam default)
LR_PYTORCH    = 1e-3     # Adam LR for the PyTorch comparison

# Gradient check
GRAD_CHECK_EPS     = 1e-5
GRAD_CHECK_TOL     = 1e-4

# Depth experiment
DEPTH_WIDTHS   = [64] * 8   # width per hidden layer for depth test
DEPTH_LIST     = [2, 4, 6, 8]


### Data Loading — Synthetic Regression

We generate a synthetic regression dataset with `sklearn.datasets.make_regression`.
The target is a noisy linear combination of 15 of the 20 input features.
After scaling, we wrap the data in a `TensorDataset` and `DataLoader` for
the PyTorch training loop (manual backprop operates directly on NumPy arrays).


In [ ]:
# ── Synthetic regression dataset ─────────────────────────────────────────────
X_np_raw, y_np_raw = make_regression(
    n_samples   = N_SAMPLES,
    n_features  = N_FEATURES,
    n_informative = N_INFORMATIVE,
    noise       = 20.0,
    random_state = SEED,
)
y_np_raw = y_np_raw.reshape(-1, 1)   # (n, 1)

# Standardise features and target for stable training
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X_np_raw).astype(np.float32)
y_scaled = scaler_y.fit_transform(y_np_raw).astype(np.float32)

# ── 80/10/10 split using PyTorch random_split ─────────────────────────────────
generator = torch.Generator().manual_seed(SEED)
full_ds = TensorDataset(
    torch.tensor(X_scaled), torch.tensor(y_scaled)
)
n_train  = int(0.8 * N_SAMPLES)
n_val    = int(0.1 * N_SAMPLES)
n_test   = N_SAMPLES - n_train - n_val
train_ds, val_ds, test_ds = torch.utils.data.random_split(
    full_ds, [n_train, n_val, n_test], generator=generator
)

# Extract numpy arrays for the manual MLP (no DataLoader needed)
train_indices = train_ds.indices
val_indices   = val_ds.indices
test_indices  = test_ds.indices

X_train_np = X_scaled[train_indices]          # (800, 20)
y_train_np = y_scaled[train_indices]          # (800, 1)
X_val_np   = X_scaled[val_indices]
y_val_np   = y_scaled[val_indices]
X_test_np  = X_scaled[test_indices]
y_test_np  = y_scaled[test_indices]

# DataLoaders for PyTorch comparison
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())

print(f"Train: {X_train_np.shape}  Val: {X_val_np.shape}  Test: {X_test_np.shape}")
print(f"y range: [{y_scaled.min():.3f}, {y_scaled.max():.3f}]")


---
## Part 1 — Chain Rule & Backprop Primitives

### The Chain Rule

For scalar composition $\mathcal{L} = f(g(x))$:

$$\frac{d\mathcal{L}}{dx} = \frac{d\mathcal{L}}{df} \cdot \frac{df}{dg} \cdot \frac{dg}{dx}$$

For a linear layer $\mathbf{Z} = \mathbf{X}\mathbf{W} + \mathbf{b}$ and loss
$\mathcal{L}(\mathbf{Z})$, the vector chain rule gives:

$$\frac{\partial\mathcal{L}}{\partial\mathbf{W}} = \mathbf{X}^\top \frac{\partial\mathcal{L}}{\partial\mathbf{Z}}, \qquad
\frac{\partial\mathcal{L}}{\partial\mathbf{X}} = \frac{\partial\mathcal{L}}{\partial\mathbf{Z}} \mathbf{W}^\top$$

Each backward function receives $\partial\mathcal{L}/\partial\text{output}$ and returns
$\partial\mathcal{L}/\partial\text{input}$ plus parameter gradients.  The intermediate
activations needed for these computations are stored in a **cache** during the forward pass.


### 1.1 Scalar Chain Rule — Worked Example

We verify the chain rule on $f(x) = (3x^2 + 2)^3$:

$$\frac{df}{dx} = 3(3x^2+2)^2 \cdot 6x = 18x(3x^2+2)^2$$

The analytical derivative is compared against a **central-difference** numerical estimate:

$$\frac{df}{dx}\bigg|_x \approx \frac{f(x+\varepsilon) - f(x-\varepsilon)}{2\varepsilon}$$


In [ ]:
# ── Scalar chain rule: f(x) = (3x^2 + 2)^3 ───────────────────────────────────

def f_scalar(x: float) -> float:
    '''Compute (3x^2 + 2)^3.'''
    return (3.0 * x ** 2 + 2.0) ** 3


def df_analytical(x: float) -> float:
    '''Analytical derivative: 18x*(3x^2+2)^2.'''
    return 18.0 * x * (3.0 * x ** 2 + 2.0) ** 2


def numerical_diff(
    func: Callable,
    x: float,
    eps: float = 1e-5,
) -> float:
    '''Central-difference numerical derivative of func at x.

    Args:
        func: Scalar function R -> R.
        x: Point at which to evaluate the derivative.
        eps: Perturbation size.

    Returns:
        Approximation of df/dx at x.
    '''
    return (func(x + eps) - func(x - eps)) / (2.0 * eps)


x_vals = np.linspace(-2.0, 2.0, 200)
df_analytic_vals  = df_analytical(x_vals)
df_numerical_vals = np.array([numerical_diff(f_scalar, xv) for xv in x_vals])
max_err = np.abs(df_analytic_vals - df_numerical_vals).max()

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_cr, axes_cr = plt.subplots(1, 2, figsize=(13, 4))
axes_cr[0].plot(x_vals, [f_scalar(xv) for xv in x_vals], color="#1f77b4", lw=2, label="f(x)")
axes_cr[0].set_xlabel("x", fontsize=11)
axes_cr[0].set_ylabel("f(x)", fontsize=11)
axes_cr[0].set_title("Function: f(x) = (3x² + 2)³", fontsize=11, fontweight="bold")
axes_cr[0].legend(fontsize=9)
axes_cr[0].grid(alpha=0.3)

axes_cr[1].plot(x_vals, df_analytic_vals,  color="#2ca02c", lw=2, label="Analytical df/dx")
axes_cr[1].plot(x_vals, df_numerical_vals, color="#d62728", lw=1.5, ls="--",
                label="Numerical df/dx")
axes_cr[1].set_xlabel("x", fontsize=11)
axes_cr[1].set_ylabel("df/dx", fontsize=11)
axes_cr[1].set_title("Chain Rule Derivative (max err = {:.2e})".format(max_err),
                      fontsize=11, fontweight="bold")
axes_cr[1].legend(fontsize=9)
axes_cr[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Max absolute error (analytical vs numerical): {max_err:.2e}")
print("✓ Chain rule derivative verified against finite differences.")


### 1.2 Layer-Level Backprop Primitives

Each layer type exposes two functions:

| Function | Input | Output | Caches |
|----------|-------|--------|--------|
| `linear_forward(X, W, b)` | $\mathbf{X}, \mathbf{W}, \mathbf{b}$ | $\mathbf{Z}$ | $\mathbf{X}, \mathbf{W}$ |
| `linear_backward(dZ, cache)` | $\partial\mathcal{L}/\partial\mathbf{Z}$ | $\partial\mathcal{L}/\partial\mathbf{X}$, $\partial\mathcal{L}/\partial\mathbf{W}$, $\partial\mathcal{L}/\partial\mathbf{b}$ | — |
| `relu_forward(Z)` | $\mathbf{Z}$ | $\mathbf{A} = \max(0, \mathbf{Z})$ | $\mathbf{Z}$ |
| `relu_backward(dA, cache)` | $\partial\mathcal{L}/\partial\mathbf{A}$ | $\partial\mathcal{L}/\partial\mathbf{Z}$ | — |
| `mse_loss(y_hat, y_true)` | predictions, targets | scalar loss | — |
| `mse_backward(y_hat, y_true)` | predictions, targets | $\partial\mathcal{L}/\partial\hat{y}$ | — |


In [ ]:
# ── Forward and backward primitives ─────────────────────────────────────────

def linear_forward(
    X: np.ndarray,
    W: np.ndarray,
    b: np.ndarray,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    '''Compute linear transformation Z = X @ W + b.

    Stores X and W in a cache dictionary for use during the backward pass.

    Args:
        X: Input array of shape (n_samples, n_in).
        W: Weight matrix of shape (n_in, n_out).
        b: Bias vector of shape (n_out,).

    Returns:
        Tuple (Z, cache) where Z has shape (n_samples, n_out)
        and cache contains keys "X" and "W".
    '''
    Z = X @ W + b
    cache: dict[str, np.ndarray] = {"X": X, "W": W}
    return Z, cache


def linear_backward(
    dZ: np.ndarray,
    cache: dict[str, np.ndarray],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Backpropagate gradients through a linear layer.

    Given dL/dZ, computes:
      dX = dZ @ W.T
      dW = X.T @ dZ / n   (mean over batch)
      db = mean(dZ, axis=0)

    Args:
        dZ: Gradient w.r.t. layer output, shape (n_samples, n_out).
        cache: Dictionary with keys "X" and "W" from linear_forward.

    Returns:
        Tuple (dX, dW, db) with shapes matching X, W, b respectively.
    '''
    X, W = cache["X"], cache["W"]
    n    = X.shape[0]
    dX   = dZ @ W.T
    dW   = X.T @ dZ / n
    db   = dZ.mean(axis=0)
    return dX, dW, db


def relu_forward(
    Z: np.ndarray,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    '''Apply ReLU activation and cache Z for backward pass.

    Args:
        Z: Pre-activation array of any shape.

    Returns:
        Tuple (A, cache) where A = max(0, Z) and cache = {"Z": Z}.
    '''
    A     = np.maximum(0.0, Z)
    cache = {"Z": Z}
    return A, cache


def relu_backward(
    dA: np.ndarray,
    cache: dict[str, np.ndarray],
) -> np.ndarray:
    '''Backpropagate gradients through ReLU activation.

    The ReLU gate passes the gradient only where the pre-activation Z > 0
    (the gate is open); elsewhere it blocks the gradient (gate is closed).

    Args:
        dA: Gradient w.r.t. ReLU output, same shape as Z.
        cache: Dictionary with key "Z" from relu_forward.

    Returns:
        dZ: Gradient w.r.t. ReLU input, same shape as dA.
    '''
    Z  = cache["Z"]
    dZ = dA * (Z > 0).astype(np.float32)
    return dZ


def mse_loss(
    y_hat: np.ndarray,
    y_true: np.ndarray,
) -> float:
    '''Compute mean squared error: mean((y_hat - y_true)^2).

    Args:
        y_hat: Predictions of shape (n_samples, 1) or (n_samples,).
        y_true: Targets of same shape as y_hat.

    Returns:
        Scalar MSE averaged over samples.
    '''
    diff = y_hat.ravel() - y_true.ravel()
    return float(np.mean(diff ** 2))


def mse_backward(
    y_hat: np.ndarray,
    y_true: np.ndarray,
) -> np.ndarray:
    '''Gradient of MSE loss w.r.t. predictions: 2*(y_hat - y_true)/n.

    Args:
        y_hat: Predictions of shape (n_samples, 1) or (n_samples,).
        y_true: Targets of same shape as y_hat.

    Returns:
        Gradient array of same shape as y_hat.
    '''
    n    = y_hat.shape[0]
    dL   = 2.0 * (y_hat.ravel() - y_true.ravel()) / n
    return dL.reshape(y_hat.shape)


# ── Quick shape sanity check ──────────────────────────────────────────────────
rng_prim = np.random.default_rng(SEED)
X_prim = rng_prim.standard_normal((8, N_FEATURES)).astype(np.float32)
W_prim = rng_prim.standard_normal((N_FEATURES, 64)).astype(np.float32) * 0.1
b_prim = np.zeros(64, dtype=np.float32)

Z_prim, cache_prim = linear_forward(X_prim, W_prim, b_prim)
A_prim, act_cache  = relu_forward(Z_prim)
print(f"linear_forward  input={X_prim.shape} -> output={Z_prim.shape}")
print(f"relu_forward    input={Z_prim.shape} -> output={A_prim.shape}")
dA_dummy = rng_prim.standard_normal(A_prim.shape).astype(np.float32)
dZ_dummy = relu_backward(dA_dummy, act_cache)
dX, dW, db = linear_backward(dZ_dummy, cache_prim)
print(f"relu_backward   dA={dA_dummy.shape} -> dZ={dZ_dummy.shape}")
print(f"linear_backward dZ={dZ_dummy.shape} -> dX={dX.shape}  dW={dW.shape}  db={db.shape}")


### 1.3 Step-by-Step Forward + Backward on One Batch

Before building the full `ManualMLP`, we trace through a two-layer network
($20 \to 64 \to 32 \to 1$) on a single mini-batch, printing each intermediate
tensor shape and gradient shape. This makes the data flow through the computational
graph concrete.


In [ ]:
# ── Two-layer MLP: step-by-step forward + backward ───────────────────────────
rng_sb = np.random.default_rng(SEED + 1)

def xavier_init(
    fan_in: int,
    fan_out: int,
    rng: np.random.Generator,
) -> np.ndarray:
    '''Xavier-uniform weight initialisation for a (fan_in, fan_out) matrix.

    Args:
        fan_in: Number of input units.
        fan_out: Number of output units.
        rng: NumPy random generator.

    Returns:
        Weight matrix of shape (fan_in, fan_out).
    '''
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return rng.uniform(-limit, limit, size=(fan_in, fan_out)).astype(np.float32)

# Parameters for a small 3-layer network: 20 -> 64 -> 32 -> 1
W1_sb = xavier_init(N_FEATURES, 64, rng_sb)
b1_sb = np.zeros(64, dtype=np.float32)
W2_sb = xavier_init(64, 32, rng_sb)
b2_sb = np.zeros(32, dtype=np.float32)
W3_sb = xavier_init(32, 1, rng_sb)
b3_sb = np.zeros(1, dtype=np.float32)

# ── FORWARD PASS ──────────────────────────────────────────────────────────────
X_sb  = X_train_np[:BATCH_SIZE]           # (64, 20)
y_sb  = y_train_np[:BATCH_SIZE]           # (64, 1)
print("=== FORWARD PASS ===")
Z1_sb, cache_lin1 = linear_forward(X_sb,  W1_sb, b1_sb)
print(f"  Layer 1 pre-act Z1:  {Z1_sb.shape}")
A1_sb, cache_act1 = relu_forward(Z1_sb)
print(f"  Layer 1 post-act A1: {A1_sb.shape}")
Z2_sb, cache_lin2 = linear_forward(A1_sb, W2_sb, b2_sb)
print(f"  Layer 2 pre-act Z2:  {Z2_sb.shape}")
A2_sb, cache_act2 = relu_forward(Z2_sb)
print(f"  Layer 2 post-act A2: {A2_sb.shape}")
Z3_sb, cache_lin3 = linear_forward(A2_sb, W3_sb, b3_sb)
print(f"  Output (y_hat):      {Z3_sb.shape}")
loss_sb = mse_loss(Z3_sb, y_sb)
print(f"  MSE loss:            {loss_sb:.6f}")

# ── BACKWARD PASS ─────────────────────────────────────────────────────────────
print("\n=== BACKWARD PASS ===")
dZ3 = mse_backward(Z3_sb, y_sb)
print(f"  dL/dZ3 (output grad): {dZ3.shape}  norm={np.linalg.norm(dZ3):.4f}")

dA2, dW3, db3 = linear_backward(dZ3, cache_lin3)
print(f"  dL/dA2:               {dA2.shape}  norm={np.linalg.norm(dA2):.4f}")
print(f"  dL/dW3:               {dW3.shape}  norm={np.linalg.norm(dW3):.4f}")

dZ2 = relu_backward(dA2, cache_act2)
print(f"  dL/dZ2 (after ReLU2): {dZ2.shape}  norm={np.linalg.norm(dZ2):.4f}")

dA1, dW2, db2 = linear_backward(dZ2, cache_lin2)
print(f"  dL/dA1:               {dA1.shape}  norm={np.linalg.norm(dA1):.4f}")
print(f"  dL/dW2:               {dW2.shape}  norm={np.linalg.norm(dW2):.4f}")

dZ1 = relu_backward(dA1, cache_act1)
print(f"  dL/dZ1 (after ReLU1): {dZ1.shape}  norm={np.linalg.norm(dZ1):.4f}")

dX_sb, dW1, db1 = linear_backward(dZ1, cache_lin1)
print(f"  dL/dX (input grad):   {dX_sb.shape}  norm={np.linalg.norm(dX_sb):.4f}")
print(f"  dL/dW1:               {dW1.shape}  norm={np.linalg.norm(dW1):.4f}")

print("\nGradient norms decrease from output to input — confirms chain rule attenuation.")


### 1.4 Computational Graph Visualisation

A **computational graph** is a directed acyclic graph (DAG) where nodes are operations and edges carry tensors.  The forward pass traces left-to-right; the backward pass reverses each edge, multiplying by the local Jacobian at each node — that multiplication is one application of the chain rule.


In [ ]:
# ── Computational graph: forward (blue) and backward (red) passes ─────────────
# We draw a simplified diagram of the three-layer MLP's computational graph,
# highlighting which arrows carry forward activations vs backward gradients.

fig_cg, ax_cg = plt.subplots(figsize=(14, 5))
ax_cg.set_xlim(0, 14)
ax_cg.set_ylim(-1, 6)
ax_cg.axis("off")
ax_cg.set_title("MLP Computational Graph: Forward (blue) and Backward (red) Passes",
                 fontsize=11, fontweight="bold")

# Node positions: (x, y) for each operation node
node_positions: dict[str, tuple[float, float]] = {
    "X":    (1.0, 2.5),
    "W1":   (2.0, 4.5),
    "b1":   (2.0, 0.5),
    "Z1":   (3.5, 2.5),
    "A1":   (5.0, 2.5),
    "W2":   (5.5, 4.5),
    "b2":   (5.5, 0.5),
    "Z2":   (7.0, 2.5),
    "A2":   (8.5, 2.5),
    "W3":   (9.0, 4.5),
    "b3":   (9.0, 0.5),
    "Z3":   (10.5, 2.5),
    "L":    (12.5, 2.5),
    "y":    (11.5, 0.5),
}

node_colors: dict[str, str] = {
    "X": "#aec7e8",  "W1": "#ffbb78", "b1": "#ffbb78",
    "Z1": "#c5e0b4", "A1": "#c5e0b4",
    "W2": "#ffbb78", "b2": "#ffbb78",
    "Z2": "#c5e0b4", "A2": "#c5e0b4",
    "W3": "#ffbb78", "b3": "#ffbb78",
    "Z3": "#c5e0b4",
    "L": "#f4a582", "y": "#aec7e8",
}

for name, (x, y) in node_positions.items():
    box = mpatches.FancyBboxPatch(
        (x - 0.4, y - 0.35), 0.8, 0.7,
        boxstyle="round,pad=0.05",
        facecolor=node_colors.get(name, "#ffffff"),
        edgecolor="k", linewidth=1.2,
    )
    ax_cg.add_patch(box)
    ax_cg.text(x, y, name, ha="center", va="center", fontsize=9, fontweight="bold")

# Forward edges (blue)
fwd_edges = [
    ("X", "Z1"), ("W1", "Z1"), ("b1", "Z1"),
    ("Z1", "A1"),
    ("A1", "Z2"), ("W2", "Z2"), ("b2", "Z2"),
    ("Z2", "A2"),
    ("A2", "Z3"), ("W3", "Z3"), ("b3", "Z3"),
    ("Z3", "L"), ("y", "L"),
]
for src, dst in fwd_edges:
    xs, ys = node_positions[src]
    xd, yd = node_positions[dst]
    ax_cg.annotate(
        "", xy=(xd - 0.4, yd), xytext=(xs + 0.4, ys),
        arrowprops=dict(arrowstyle="->", color="#1f77b4", lw=1.5),
    )

# Backward edges (red, slightly offset)
bwd_edges = [
    ("L", "Z3"), ("Z3", "A2"), ("Z3", "W3"), ("Z3", "b3"),
    ("A2", "Z2"), ("Z2", "A1"), ("Z2", "W2"), ("Z2", "b2"),
    ("A1", "Z1"), ("Z1", "X"),  ("Z1", "W1"), ("Z1", "b1"),
]
for src, dst in bwd_edges:
    xs, ys = node_positions[src]
    xd, yd = node_positions[dst]
    ax_cg.annotate(
        "", xy=(xd + 0.4, yd - 0.15), xytext=(xs - 0.4, ys - 0.15),
        arrowprops=dict(arrowstyle="->", color="#d62728", lw=1.2, linestyle="dashed"),
    )

fwd_patch = mpatches.Patch(color="#1f77b4", label="Forward pass (activations)")
bwd_patch = mpatches.Patch(color="#d62728", label="Backward pass (gradients)")
ax_cg.legend(handles=[fwd_patch, bwd_patch], loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()
print("Blue arrows = forward pass (values flow left to right).")
print("Red  arrows = backward pass (gradients flow right to left).")
print("Each backward arrow applies one step of the chain rule.")


---
## Part 2 — ManualMLP: Full Backpropagation Class

We now encapsulate the primitives into a `ManualMLP` class.  The class supports:

- **Arbitrary depth** via the `layer_sizes` list `[n_in, h1, h2, ..., n_out]`.
- **Forward pass** stores a per-layer cache for backward.
- **Backward pass** iterates layers in reverse, accumulating $\partial\mathcal{L}/\partial\mathbf{W}_l$ and $\partial\mathcal{L}/\partial\mathbf{b}_l$.
- **SGD update** applies $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$.


In [ ]:
# ── ManualMLP: MLP with explicit backpropagation ─────────────────────────────

class ManualMLP:
    '''Multi-layer perceptron with manual forward and backward passes.

    Implements a fully-connected network without PyTorch autograd.
    ReLU activations are applied after every hidden layer; the output
    layer produces a raw linear value (suitable for regression with MSE).

    Attributes:
        layer_sizes: Architecture as [n_in, h1, ..., n_out].
        n_layers:    Number of weight matrices (= len(layer_sizes) - 1).
        weights:     List of weight matrices W_l, shape (n_in_l, n_out_l).
        biases:      List of bias vectors b_l, shape (n_out_l,).
        grads_W:     Gradient of loss w.r.t. each weight matrix.
        grads_b:     Gradient of loss w.r.t. each bias vector.
    '''

    def __init__(
        self,
        layer_sizes: list[int],
        seed: int = SEED,
    ) -> None:
        '''Initialise weights with Xavier-uniform and biases to zero.

        Args:
            layer_sizes: List [n_in, h1, ..., n_out] of layer widths.
            seed: Random seed for reproducible initialisation.
        '''
        rng = np.random.default_rng(seed)
        self.layer_sizes = layer_sizes
        self.n_layers    = len(layer_sizes) - 1
        self.weights:  list[np.ndarray] = []
        self.biases:   list[np.ndarray] = []
        self.grads_W:  list[np.ndarray] = []
        self.grads_b:  list[np.ndarray] = []
        self._caches:  list[dict]        = []

        for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            limit = np.sqrt(6.0 / (fan_in + fan_out))
            W = rng.uniform(-limit, limit, size=(fan_in, fan_out)).astype(np.float32)
            b = np.zeros(fan_out, dtype=np.float32)
            self.weights.append(W)
            self.biases.append(b)
            self.grads_W.append(np.zeros_like(W))
            self.grads_b.append(np.zeros_like(b))

    def forward(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        '''Execute the forward pass and cache all intermediate activations.

        Applies ReLU after every hidden layer; the output layer is linear.

        Args:
            X: Input array of shape (n_samples, n_in).

        Returns:
            Predictions of shape (n_samples, n_out).
        '''
        self._caches = []
        current      = X
        for layer_idx in range(self.n_layers):
            Z, lin_cache = linear_forward(
                current, self.weights[layer_idx], self.biases[layer_idx]
            )
            is_hidden = layer_idx < self.n_layers - 1
            if is_hidden:
                A, act_cache = relu_forward(Z)
                self._caches.append({"lin": lin_cache, "act": act_cache, "has_act": True})
                current = A
            else:
                self._caches.append({"lin": lin_cache, "has_act": False})
                current = Z
        return current

    def backward(
        self,
        y_hat: np.ndarray,
        y_true: np.ndarray,
    ) -> float:
        '''Execute the backward pass and store gradients.

        Computes dL/dW_l and dL/db_l for every layer using the chain rule,
        backpropagating from the output loss to the first layer.

        Args:
            y_hat:  Predictions from the most recent forward call.
            y_true: Ground-truth targets of same shape as y_hat.

        Returns:
            Scalar MSE loss value for the current batch.
        '''
        loss = mse_loss(y_hat, y_true)
        dA   = mse_backward(y_hat, y_true)
        for layer_idx in reversed(range(self.n_layers)):
            cache = self._caches[layer_idx]
            dZ    = relu_backward(dA, cache["act"]) if cache["has_act"] else dA
            dA, dW, db               = linear_backward(dZ, cache["lin"])
            self.grads_W[layer_idx]  = dW
            self.grads_b[layer_idx]  = db
        return loss

    def update(
        self,
        learning_rate: float,
    ) -> None:
        '''Apply vanilla SGD update using the stored gradients.

        Args:
            learning_rate: Step size for the gradient descent update.
        '''
        for layer_idx in range(self.n_layers):
            self.weights[layer_idx] -= learning_rate * self.grads_W[layer_idx]
            self.biases[layer_idx]  -= learning_rate * self.grads_b[layer_idx]

    def predict(
        self,
        X: np.ndarray,
    ) -> np.ndarray:
        '''Run inference (forward pass without gradient caching).

        Args:
            X: Input array of shape (n_samples, n_in).

        Returns:
            Predictions of shape (n_samples, n_out).
        '''
        return self.forward(X)

    def grad_norms(self) -> list[float]:
        '''Return the Frobenius norm of each weight gradient matrix.

        Returns:
            List of gradient norms, one per layer.
        '''
        return [float(np.linalg.norm(gW)) for gW in self.grads_W]


# ── Quick construction test ───────────────────────────────────────────────────
arch = [N_FEATURES] + HIDDEN_SIZES + [OUTPUT_SIZE]  # [20, 64, 32, 1]
model_manual = ManualMLP(layer_sizes=arch)
y_hat_test = model_manual.forward(X_train_np[:4])
print(f"Architecture: {arch}")
print(f"Forward pass output shape: {y_hat_test.shape}  (expected (4, 1))")


### 2.1 Numerical Gradient Check

A **gradient check** is the standard way to verify backpropagation implementations.
For every parameter $\theta_i$ we estimate the gradient via central differences:

$$\hat{g}_i = \frac{\mathcal{L}(\theta_i + \varepsilon) - \mathcal{L}(\theta_i - \varepsilon)}{2\varepsilon}$$

and compare with the analytical gradient $g_i = \partial\mathcal{L}/\partial\theta_i$
computed by backprop.  The **relative error**:

$$e = \frac{\|\hat{g} - g\|_\infty}{\|\hat{g}\|_\infty + \|g\|_\infty + 10^{-8}}$$

should be $< 10^{-4}$ for a correct implementation with `float32` and $\varepsilon = 10^{-5}$.


In [ ]:
# ── Numerical gradient checker ────────────────────────────────────────────────

def numerical_gradient_check(
    model_gc: ManualMLP,
    X_batch: np.ndarray,
    y_batch: np.ndarray,
    layer_idx: int,
    param_type: str,
    eps: float = GRAD_CHECK_EPS,
) -> tuple[np.ndarray, np.ndarray, float]:
    '''Compare analytical backprop gradient against finite-difference estimate.

    Perturbs each element of the target parameter individually,
    re-runs the forward pass, and estimates the gradient via central differences.

    Args:
        model_gc:   ManualMLP whose gradients we are checking.
        X_batch:    Small input batch for speed.
        y_batch:    Corresponding targets.
        layer_idx:  Index of the layer whose parameter to check.
        param_type: Either "W" (weight matrix) or "b" (bias vector).
        eps:        Finite-difference step size.

    Returns:
        Tuple of (analytical_grad, numerical_grad, relative_error).
    '''
    param = model_gc.weights[layer_idx] if param_type == "W" else model_gc.biases[layer_idx]
    numerical = np.zeros_like(param)
    it = np.nditer(param, flags=["multi_index"])
    while not it.finished:
        idx = it.multi_index
        orig = float(param[idx])
        param[idx] = orig + eps
        loss_plus = mse_loss(model_gc.forward(X_batch), y_batch)
        param[idx] = orig - eps
        loss_minus = mse_loss(model_gc.forward(X_batch), y_batch)
        numerical[idx] = (loss_plus - loss_minus) / (2.0 * eps)
        param[idx] = orig
        it.iternext()

    # Run one forward+backward to get analytical gradient
    y_hat_gc = model_gc.forward(X_batch)
    model_gc.backward(y_hat_gc, y_batch)
    analytical = (
        model_gc.grads_W[layer_idx].copy() if param_type == "W"
        else model_gc.grads_b[layer_idx].copy()
    )
    denom   = np.abs(analytical).max() + np.abs(numerical).max() + 1e-8
    rel_err = float(np.abs(analytical - numerical).max() / denom)
    return analytical, numerical, rel_err


# ── Run gradient check on a small batch ──────────────────────────────────────
X_gc = X_train_np[:12]
y_gc = y_train_np[:12]
model_gc = ManualMLP(layer_sizes=[N_FEATURES, 16, 8, 1])  # small, fast

print("Gradient check results (max relative error per layer):")
print(f"  {'Layer':>5s}  {'Param':>5s}  {'Rel Error':>12s}  {'Status':>8s}")
print("  " + "-" * 38)
for layer_idx in range(model_gc.n_layers):
    for ptype in ["W", "b"]:
        _, _, rel_err = numerical_gradient_check(model_gc, X_gc, y_gc, layer_idx, ptype)
        status = "PASS" if rel_err < GRAD_CHECK_TOL else "FAIL"
        print(f"  {layer_idx + 1:>5d}  {ptype:>5s}  {rel_err:>12.2e}  {status:>8s}")

print("\nAll errors < {:.0e} => backpropagation implementation is correct.".format(GRAD_CHECK_TOL))


---
## Part 3 — Training & PyTorch Comparison

### 3.1 Training with Manual Backprop

We now train `ManualMLP` on the synthetic regression dataset using mini-batch
gradient descent. The training loop follows the standard pattern:
`forward → loss → backward → update`.

We track the gradient norm of each layer's weight matrix across training to
observe how information flows (or fails to flow) from output to input.


In [ ]:
# ── Mini-batch training with ManualMLP ───────────────────────────────────────

def train_manual_mlp(
    model_t: ManualMLP,
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_vl: np.ndarray,
    y_vl: np.ndarray,
    num_epochs: int,
    batch_size: int,
    learning_rate: float,
    rng_tr: np.random.Generator,
) -> tuple[list[float], list[float], dict[int, list[float]]]:
    '''Train ManualMLP with mini-batch SGD and track gradient norms.

    Args:
        model_t:       ManualMLP instance to train.
        X_tr:          Training inputs, shape (n_train, n_in).
        y_tr:          Training targets, shape (n_train, 1).
        X_vl:          Validation inputs.
        y_vl:          Validation targets.
        num_epochs:    Number of full passes over the training set.
        batch_size:    Number of samples per mini-batch.
        learning_rate: SGD learning rate.
        rng_tr:        Random generator for shuffle.

    Returns:
        Tuple of (train_losses, val_losses, grad_norm_history) where
        grad_norm_history maps layer index to list of per-batch gradient norms.
    '''
    n_train    = X_tr.shape[0]
    train_losses: list[float] = []
    val_losses:   list[float] = []
    grad_norms_hist: dict[int, list[float]] = {
        l: [] for l in range(model_t.n_layers)
    }

    for epoch in range(num_epochs):
        perm    = rng_tr.permutation(n_train)
        X_shuf  = X_tr[perm]
        y_shuf  = y_tr[perm]
        ep_loss = 0.0
        n_batches = 0
        t_start = time.time()
        for start in range(0, n_train, batch_size):
            X_b    = X_shuf[start : start + batch_size]
            y_b    = y_shuf[start : start + batch_size]
            y_hat  = model_t.forward(X_b)
            loss_b = model_t.backward(y_hat, y_b)
            for lyr_idx in range(model_t.n_layers):
                grad_norms_hist[lyr_idx].append(
                    float(np.linalg.norm(model_t.grads_W[lyr_idx]))
                )
            model_t.update(learning_rate)
            ep_loss   += loss_b
            n_batches += 1

        val_pred   = model_t.predict(X_vl)
        val_loss   = mse_loss(val_pred, y_vl)
        ep_loss_avg = ep_loss / n_batches
        train_losses.append(ep_loss_avg)
        val_losses.append(val_loss)
        elapsed = time.time() - t_start
        if (epoch + 1) % 5 == 0 or epoch == 0:
            val_rmse = float(np.sqrt(val_loss))
            print(f"Epoch {epoch+1:>2d}/{num_epochs} | "
                  f"Train Loss: {ep_loss_avg:.4f} | Val Loss: {val_loss:.4f} | "
                  f"Val RMSE: {val_rmse:.4f} | Time: {elapsed:.1f}s")

    return train_losses, val_losses, grad_norms_hist


rng_train = np.random.default_rng(SEED)
t0 = time.time()
manual_train_losses, manual_val_losses, manual_grad_norms = train_manual_mlp(
    model_manual, X_train_np, y_train_np, X_val_np, y_val_np,
    NUM_EPOCHS, BATCH_SIZE, LEARNING_RATE, rng_train
)
print(f"\nTotal training time: {time.time() - t0:.1f}s")
test_loss_manual = mse_loss(model_manual.predict(X_test_np), y_test_np)
print(f"Test MSE  (ManualMLP):  {test_loss_manual:.4f}")
print(f"Test RMSE (ManualMLP):  {float(np.sqrt(test_loss_manual)):.4f}")


### 3.1b Learning Rate Sensitivity

The learning rate $\eta$ is the single most important hyperparameter for gradient descent.  Too small: slow convergence.  Too large: divergence (loss goes to $\infty$).  We sweep six values and compare convergence on the validation set.


In [ ]:
# ── Learning rate sensitivity: final val MSE across LR values ─────────────────
# A key practical insight: too-small LR -> slow convergence; too-large LR -> divergence.
# We sweep 6 LR values and train each ManualMLP for NUM_EPOCHS epochs.

lr_sweep: list[float] = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2]
lr_results: list[dict] = []

for lr_val in lr_sweep:
    m_lr = ManualMLP(layer_sizes=arch)
    rng_lr = np.random.default_rng(SEED + 42)
    n_tr   = X_train_np.shape[0]
    tr_losses_lr: list[float] = []
    vl_losses_lr: list[float] = []

    for epoch in range(NUM_EPOCHS):
        perm_lr = rng_lr.permutation(n_tr)
        X_shuf_lr = X_train_np[perm_lr]
        y_shuf_lr = y_train_np[perm_lr]
        ep_loss_lr = 0.0
        n_b_lr = 0
        for start in range(0, n_tr, BATCH_SIZE):
            X_b_lr = X_shuf_lr[start : start + BATCH_SIZE]
            y_b_lr = y_shuf_lr[start : start + BATCH_SIZE]
            y_hat_lr = m_lr.forward(X_b_lr)
            loss_b_lr = m_lr.backward(y_hat_lr, y_b_lr)
            m_lr.update(lr_val)
            ep_loss_lr += loss_b_lr
            n_b_lr += 1
        vl_pred_lr  = m_lr.predict(X_val_np)
        vl_loss_lr  = mse_loss(vl_pred_lr, y_val_np)
        tr_losses_lr.append(ep_loss_lr / n_b_lr)
        vl_losses_lr.append(vl_loss_lr)

    final_val = vl_losses_lr[-1]
    diverged  = float("inf") if np.isnan(final_val) or final_val > 1e4 else final_val
    lr_results.append({
        "LR":      lr_val,
        "Train":   tr_losses_lr,
        "Val":     vl_losses_lr,
        "FinalVal": diverged,
    })
    status = "DIVERGED" if diverged == float("inf") else f"{diverged:.5f}"
    print(f"  LR={lr_val:.0e}  Final val MSE = {status}")

# ── Plot convergence curves ───────────────────────────────────────────────────
fig_lr, axes_lr = plt.subplots(1, 2, figsize=(13, 4))
cmap_lr = plt.cm.viridis
epochs_lr = list(range(1, NUM_EPOCHS + 1))

for i, result in enumerate(lr_results):
    color_lr = cmap_lr(i / len(lr_results))
    lbl = f"LR={result['LR']:.0e}"
    tr_arr = [min(v, 10.0) for v in result["Train"]]
    vl_arr = [min(v, 10.0) for v in result["Val"]]
    axes_lr[0].plot(epochs_lr, tr_arr, color=color_lr, lw=1.5, label=lbl)
    axes_lr[1].plot(epochs_lr, vl_arr, color=color_lr, lw=1.5, label=lbl)

axes_lr[0].set_xlabel("Epoch", fontsize=11)
axes_lr[0].set_ylabel("Train MSE (clipped at 10)", fontsize=11)
axes_lr[0].set_title("Learning Rate Sensitivity: Train Loss", fontsize=11, fontweight="bold")
axes_lr[0].legend(fontsize=8)
axes_lr[0].grid(alpha=0.3)

axes_lr[1].set_xlabel("Epoch", fontsize=11)
axes_lr[1].set_ylabel("Val MSE (clipped at 10)", fontsize=11)
axes_lr[1].set_title("Learning Rate Sensitivity: Val Loss", fontsize=11, fontweight="bold")
axes_lr[1].legend(fontsize=8)
axes_lr[1].grid(alpha=0.3)

plt.suptitle("ManualMLP: LR Sensitivity (SGD)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

best_lr_row = min([r for r in lr_results if r["FinalVal"] < 1e4],
                  key=lambda r: r["FinalVal"])
print(f"\nBest LR: {best_lr_row['LR']:.0e}  Final val MSE: {best_lr_row['FinalVal']:.5f}")


### 3.2 PyTorch Autograd Comparison

We train the **identical architecture** using PyTorch's `nn.Linear` + `nn.ReLU`
and Adam optimiser.  The expected outcome: similar or better final loss
(Adam adapts the learning rate per-parameter; our manual loop uses vanilla SGD).

> **05-07 note:** We use PyTorch autograd as a black box here.  The internal mechanics
> — `grad_fn`, computation graph, `torch.autograd.Function` — are covered in **05-07**.


In [ ]:
# ── PyTorch MLP: same architecture ───────────────────────────────────────────

class PyTorchMLP(nn.Module):
    '''Fully-connected MLP built from nn.Linear and nn.ReLU.

    Mirrors the ManualMLP architecture for a fair comparison.

    Attributes:
        net: Sequential stack of linear layers and ReLU activations.
    '''

    def __init__(
        self,
        layer_sizes: list[int],
    ) -> None:
        '''Build the MLP as a sequential module.

        Args:
            layer_sizes: List [n_in, h1, ..., n_out] of layer widths.
        '''
        super().__init__()
        layers: list[nn.Module] = []
        for fan_in, fan_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            layers.append(nn.Linear(fan_in, fan_out))
            if fan_out != layer_sizes[-1]:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        '''Forward pass through the MLP.

        Args:
            x: Input tensor of shape (batch_size, n_in).

        Returns:
            Output tensor of shape (batch_size, n_out).
        '''
        return self.net(x)


def train_pytorch_mlp(
    model_pt: PyTorchMLP,
    train_ldr: DataLoader,
    val_ldr: DataLoader,
    num_epochs: int,
    lr: float,
) -> tuple[list[float], list[float]]:
    '''Train a PyTorch regression MLP with Adam and MSELoss.

    Args:
        model_pt:   PyTorchMLP instance.
        train_ldr:  Training DataLoader.
        val_ldr:    Validation DataLoader.
        num_epochs: Number of training epochs.
        lr:         Adam learning rate.

    Returns:
        Tuple of (train_losses, val_losses) per epoch.
    '''
    model_pt.to(device)
    optimizer_pt = optim.Adam(model_pt.parameters(), lr=lr)
    criterion_pt = nn.MSELoss()
    train_losses_pt: list[float] = []
    val_losses_pt:   list[float] = []

    for epoch in range(num_epochs):
        model_pt.train()
        ep_loss = 0.0
        n_batches = 0
        t_start = time.time()
        for X_b, y_b in train_ldr:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer_pt.zero_grad()
            y_hat_pt = model_pt(X_b)
            loss_pt  = criterion_pt(y_hat_pt, y_b)
            loss_pt.backward()
            optimizer_pt.step()
            ep_loss   += loss_pt.item()
            n_batches += 1
        model_pt.eval()
        val_loss_pt = 0.0
        with torch.no_grad():
            for X_vb, y_vb in val_ldr:
                X_vb, y_vb = X_vb.to(device), y_vb.to(device)
                val_loss_pt += criterion_pt(model_pt(X_vb), y_vb).item()
        ep_avg  = ep_loss / n_batches
        vl_avg  = val_loss_pt / len(val_ldr)
        train_losses_pt.append(ep_avg)
        val_losses_pt.append(vl_avg)
        elapsed = time.time() - t_start
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:>2d}/{num_epochs} | "
                  f"Train Loss: {ep_avg:.4f} | Val Loss: {vl_avg:.4f} | "
                  f"Val RMSE: {float(np.sqrt(vl_avg)):.4f} | Time: {elapsed:.1f}s")
    return train_losses_pt, val_losses_pt


torch.manual_seed(SEED)
model_pt = PyTorchMLP(layer_sizes=arch)
pt_train_losses, pt_val_losses = train_pytorch_mlp(
    model_pt, train_loader, val_loader, NUM_EPOCHS, LR_PYTORCH
)
pt_test_loss = nn.MSELoss()(
    model_pt(torch.tensor(X_test_np).to(device)).cpu(),
    torch.tensor(y_test_np)
).item()
print(f"\nTest MSE  (PyTorchMLP): {pt_test_loss:.4f}")
print(f"Test RMSE (PyTorchMLP): {float(np.sqrt(pt_test_loss)):.4f}")


In [ ]:
# ── Training curve comparison ─────────────────────────────────────────────────
fig_cmp, axes_cmp = plt.subplots(1, 2, figsize=(13, 4))

epochs_ax = list(range(1, NUM_EPOCHS + 1))
axes_cmp[0].plot(epochs_ax, manual_train_losses, color="#1f77b4", lw=2,
                 label="ManualMLP train (SGD)")
axes_cmp[0].plot(epochs_ax, manual_val_losses,   color="#1f77b4", lw=2, ls="--",
                 label="ManualMLP val")
axes_cmp[0].plot(epochs_ax, pt_train_losses,     color="#d62728", lw=2,
                 label="PyTorchMLP train (Adam)")
axes_cmp[0].plot(epochs_ax, pt_val_losses,       color="#d62728", lw=2, ls="--",
                 label="PyTorchMLP val")
axes_cmp[0].set_xlabel("Epoch", fontsize=11)
axes_cmp[0].set_ylabel("MSE Loss", fontsize=11)
axes_cmp[0].set_title("Training Curves: Manual vs PyTorch", fontsize=11, fontweight="bold")
axes_cmp[0].legend(fontsize=8)
axes_cmp[0].grid(alpha=0.3)

comparison_rows = [
    {"Model": "ManualMLP (SGD)",   "Optimizer": "SGD",
     "Final Train MSE": manual_train_losses[-1], "Val MSE": manual_val_losses[-1],
     "Test MSE": test_loss_manual},
    {"Model": "PyTorchMLP (Adam)", "Optimizer": "Adam",
     "Final Train MSE": pt_train_losses[-1], "Val MSE": pt_val_losses[-1],
     "Test MSE": pt_test_loss},
]
cmp_df = pd.DataFrame(comparison_rows)
print(cmp_df.to_string(index=False, float_format="{:.5f}".format))

y_pred_manual = model_manual.predict(X_test_np).ravel()
y_true_test   = y_test_np.ravel()
axes_cmp[1].scatter(y_true_test, y_pred_manual, s=15, alpha=0.5, color="#1f77b4",
                    label="ManualMLP")
y_pred_pt = model_pt(torch.tensor(X_test_np).to(device)).detach().cpu().numpy().ravel()
axes_cmp[1].scatter(y_true_test, y_pred_pt, s=15, alpha=0.5, color="#d62728",
                    label="PyTorchMLP", marker="^")
lim_min = min(y_true_test.min(), y_pred_manual.min(), y_pred_pt.min())
lim_max = max(y_true_test.max(), y_pred_manual.max(), y_pred_pt.max())
axes_cmp[1].plot([lim_min, lim_max], [lim_min, lim_max], "k--", lw=1, label="Perfect fit")
axes_cmp[1].set_xlabel("True y (standardised)", fontsize=11)
axes_cmp[1].set_ylabel("Predicted y (standardised)", fontsize=11)
axes_cmp[1].set_title("Predicted vs True on Test Set", fontsize=11, fontweight="bold")
axes_cmp[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


---
## Part 4 — Gradient Flow & Depth Effects

### The Vanishing Gradient Preview

When gradients are backpropagated through many layers, the chain rule multiplies
many Jacobians together.  If the spectral norm of each Jacobian is $< 1$, the
gradient magnitude shrinks exponentially with depth (vanishing gradient).  If $> 1$,
it explodes.

We visualise two phenomena:
1. **Per-layer gradient norms across training** — how active is each layer's learning?
2. **Depth experiment** — with the same data and architecture width, how does the
   gradient at the first layer scale with the number of hidden layers?

> Vanishing gradients motivated the design of ReLU activations, skip connections,
> and careful weight initialisation (see **05-08**).


In [ ]:
# ── Gradient norm per layer across training ───────────────────────────────────
fig_gf, axes_gf = plt.subplots(1, model_manual.n_layers, figsize=(14, 3.5), sharey=False)

colors_layers = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for lyr_idx in range(model_manual.n_layers):
    norms = manual_grad_norms[lyr_idx]
    batches_per_epoch = int(np.ceil(len(X_train_np) / BATCH_SIZE))
    n_steps = len(norms)
    axes_gf[lyr_idx].plot(range(n_steps), norms,
                           color=colors_layers[lyr_idx % len(colors_layers)], lw=0.9)
    axes_gf[lyr_idx].set_title(f"Layer {lyr_idx + 1}\n({arch[lyr_idx]}->{arch[lyr_idx+1]})",
                                fontsize=9, fontweight="bold")
    axes_gf[lyr_idx].set_xlabel("Training step", fontsize=8)
    axes_gf[lyr_idx].set_ylabel("||dL/dW||_F", fontsize=8)
    axes_gf[lyr_idx].tick_params(labelsize=7)
    axes_gf[lyr_idx].grid(alpha=0.3)
    mean_norm = float(np.mean(norms))
    axes_gf[lyr_idx].axhline(mean_norm, color="k", lw=0.8, ls="--",
                              label=f"mean={mean_norm:.4f}")
    axes_gf[lyr_idx].legend(fontsize=7)

plt.suptitle("Gradient Norm per Layer Across Training (ManualMLP)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Summary statistics ────────────────────────────────────────────────────────
print("Mean gradient norm per layer (averaged across all training steps):")
for lyr_idx in range(model_manual.n_layers):
    norms    = manual_grad_norms[lyr_idx]
    mean_n   = float(np.mean(norms))
    std_n    = float(np.std(norms))
    print(f"  Layer {lyr_idx + 1} ({arch[lyr_idx]:>3d}->{arch[lyr_idx+1]:>3d}): "
          f"mean={mean_n:.6f}  std={std_n:.6f}")
print("\nObservation: earlier (input-side) layers typically have smaller gradients.")


In [ ]:
# ── Depth experiment: gradient attenuation vs number of layers ────────────────
# We build networks of increasing depth (all same width), run one forward+backward
# pass on a fixed random batch, and measure the gradient norm at the FIRST layer.

def first_layer_grad_norm(
    n_hidden: int,
    width: int,
    X_batch: np.ndarray,
    y_batch: np.ndarray,
    seed: int = SEED,
) -> float:
    '''Build a ManualMLP with n_hidden hidden layers, run one backward pass,
    and return the gradient norm at the first layer.

    Args:
        n_hidden:  Number of hidden layers.
        width:     Width of each hidden layer.
        X_batch:   Input batch of shape (n, n_in).
        y_batch:   Target batch of shape (n, 1).
        seed:      Random seed for weight initialisation.

    Returns:
        Frobenius norm of the first-layer weight gradient.
    '''
    sizes = [X_batch.shape[1]] + [width] * n_hidden + [1]
    m     = ManualMLP(layer_sizes=sizes, seed=seed)
    y_hat = m.forward(X_batch)
    m.backward(y_hat, y_batch)
    return float(np.linalg.norm(m.grads_W[0]))


rng_depth = np.random.default_rng(SEED + 5)
X_depth   = rng_depth.standard_normal((BATCH_SIZE, N_FEATURES)).astype(np.float32)
y_depth   = rng_depth.standard_normal((BATCH_SIZE, 1)).astype(np.float32)

depth_rows: list[dict] = []
print("First-layer gradient norm vs network depth:")
print(f"  {'Depth':>5s}  {'Grad Norm':>12s}  {'Ratio to depth-2':>18s}")
norm_at_2 = first_layer_grad_norm(2, 64, X_depth, y_depth)
for n_hidden in DEPTH_LIST:
    norm_d = first_layer_grad_norm(n_hidden, 64, X_depth, y_depth)
    ratio  = norm_d / (norm_at_2 + 1e-12)
    depth_rows.append({"Hidden layers": n_hidden, "Grad norm": norm_d, "Ratio": ratio})
    print(f"  {n_hidden:>5d}  {norm_d:>12.6f}  {ratio:>18.4f}")

fig_dep, axes_dep = plt.subplots(1, 2, figsize=(13, 4))
depths     = [r["Hidden layers"] for r in depth_rows]
norms_dep  = [r["Grad norm"] for r in depth_rows]
ratios_dep = [r["Ratio"] for r in depth_rows]

axes_dep[0].bar(depths, norms_dep, color="#1f77b4", edgecolor="k", lw=0.8)
axes_dep[0].set_xlabel("Number of hidden layers", fontsize=11)
axes_dep[0].set_ylabel("||dL/dW_1||_F (first layer)", fontsize=11)
axes_dep[0].set_title("Gradient Norm at First Layer vs Depth",
                       fontsize=11, fontweight="bold")
axes_dep[0].set_xticks(depths)
axes_dep[0].grid(axis="y", alpha=0.3)

axes_dep[1].plot(depths, ratios_dep, "o-", color="#d62728", lw=2, markersize=8)
axes_dep[1].axhline(1.0, color="k", lw=1, ls="--", label="Depth-2 baseline")
axes_dep[1].set_xlabel("Number of hidden layers", fontsize=11)
axes_dep[1].set_ylabel("Ratio relative to depth-2", fontsize=11)
axes_dep[1].set_title("Relative Gradient Attenuation with Depth",
                       fontsize=11, fontweight="bold")
axes_dep[1].legend(fontsize=9)
axes_dep[1].grid(alpha=0.3)

plt.suptitle("Vanishing Gradient Preview: Deeper Networks, Smaller First-Layer Gradients",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()
print("\nWith ReLU activations, gradients do not vanish as severely as with Sigmoid/Tanh,")
print("but attenuation still occurs due to the ReLU dead-neuron effect at initialisation.")


### 4.2 Gradient Clipping

When gradients are very large (explosion), a single update can overshoot and destabilise training.  **Gradient clipping** rescales the gradient vector so its global $L_2$ norm does not exceed a threshold $\tau$:

$$\mathbf{g} \leftarrow \mathbf{g} \cdot \min\!\left(1,\, \frac{\tau}{\|\mathbf{g}\|}\right)$$

We demonstrate the effect on our `ManualMLP` using an aggressively large learning rate.


In [ ]:
# ── Gradient clipping: prevent explosion in deep networks ─────────────────────
# Gradient clipping rescales the gradient vector if its norm exceeds a threshold:
#   if ||g|| > max_norm:  g <- g * (max_norm / ||g||)
# This is standard practice in RNN training and useful whenever gradients explode.

def clip_gradients(
    model_clip: ManualMLP,
    max_norm: float,
) -> float:
    '''Clip all weight gradients so the global gradient norm <= max_norm.

    Computes the global norm across all parameter gradient matrices and
    rescales every gradient proportionally if the norm exceeds max_norm.

    Args:
        model_clip: ManualMLP whose grads_W lists will be clipped in-place.
        max_norm:   Maximum allowed global gradient norm.

    Returns:
        The global gradient norm BEFORE clipping.
    '''
    global_norm = float(np.sqrt(sum(
        float(np.sum(gW ** 2)) for gW in model_clip.grads_W
    )))
    if global_norm > max_norm:
        scale = max_norm / (global_norm + 1e-8)
        for lyr_idx in range(model_clip.n_layers):
            model_clip.grads_W[lyr_idx] *= scale
            model_clip.grads_b[lyr_idx] *= scale
    return global_norm


# ── Compare training with and without gradient clipping ───────────────────────
# Use an aggressive LR to trigger gradient explosion, then observe clipping effect.
CLIP_LR      = 5e-2
MAX_NORM     = 1.0
clip_configs = {
    "No clipping":  False,
    "Clip norm=1.0": True,
}
clip_histories: dict[str, list[float]] = {}

rng_clip = np.random.default_rng(SEED + 7)
for label, do_clip in clip_configs.items():
    m_clip = ManualMLP(layer_sizes=arch, seed=SEED)
    rng_c  = np.random.default_rng(SEED + 7)
    n_tr   = X_train_np.shape[0]
    vl_curve: list[float] = []
    for epoch in range(NUM_EPOCHS):
        perm_c = rng_c.permutation(n_tr)
        X_shuf_c = X_train_np[perm_c]
        y_shuf_c = y_train_np[perm_c]
        for start in range(0, n_tr, BATCH_SIZE):
            X_b_c = X_shuf_c[start : start + BATCH_SIZE]
            y_b_c = y_shuf_c[start : start + BATCH_SIZE]
            y_hat_c = m_clip.forward(X_b_c)
            m_clip.backward(y_hat_c, y_b_c)
            if do_clip:
                clip_gradients(m_clip, MAX_NORM)
            m_clip.update(CLIP_LR)
        vl_pred_c = m_clip.predict(X_val_np)
        vl_loss_c = mse_loss(vl_pred_c, y_val_np)
        vl_curve.append(min(vl_loss_c, 20.0))  # clip display to 20
    clip_histories[label] = vl_curve

fig_clp, ax_clp = plt.subplots(figsize=(9, 4))
colors_clip = {"No clipping": "#d62728", "Clip norm=1.0": "#2ca02c"}
for label, curve in clip_histories.items():
    ax_clp.plot(range(1, NUM_EPOCHS + 1), curve, lw=2, label=label,
                color=colors_clip[label])
ax_clp.set_xlabel("Epoch", fontsize=11)
ax_clp.set_ylabel("Val MSE (clipped display at 20)", fontsize=11)
ax_clp.set_title(f"Gradient Clipping Demo (LR={CLIP_LR:.0e}, max_norm={MAX_NORM})",
                  fontsize=11, fontweight="bold")
ax_clp.legend(fontsize=9)
ax_clp.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for label, curve in clip_histories.items():
    final = curve[-1]
    print(f"  {label:<22s}  Final val MSE = {final:.5f}")
print("\nGradient clipping prevents divergence when using large learning rates.")
print("It is especially important in RNN/LSTM training (Module 07).")


In [ ]:
# ── Gradient statistics: magnitude distribution across layers ─────────────────
# After full training, run one backward pass and compare gradient value distributions

y_hat_final  = model_manual.forward(X_train_np[:200])
model_manual.backward(y_hat_final, y_train_np[:200])

fig_gd, axes_gd = plt.subplots(1, model_manual.n_layers, figsize=(14, 4), sharey=False)
for lyr_idx in range(model_manual.n_layers):
    gw_flat = model_manual.grads_W[lyr_idx].ravel()
    axes_gd[lyr_idx].hist(gw_flat, bins=50, color=colors_layers[lyr_idx % 3],
                           edgecolor="none", density=True, alpha=0.8)
    axes_gd[lyr_idx].axvline(float(gw_flat.mean()), color="k", lw=1.5, ls="--",
                              label=f"mean={float(gw_flat.mean()):.4f}")
    axes_gd[lyr_idx].set_title(f"Layer {lyr_idx + 1} grad dist",
                                fontsize=9, fontweight="bold")
    axes_gd[lyr_idx].set_xlabel("dL/dW value", fontsize=8)
    axes_gd[lyr_idx].set_ylabel("Density", fontsize=8)
    axes_gd[lyr_idx].legend(fontsize=7)
    axes_gd[lyr_idx].tick_params(labelsize=7)
plt.suptitle("Gradient Value Distributions Across Layers (after training)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Weight update magnitude: gradient vs weight ratio ─────────────────────────
print("Gradient-to-weight ratio ||dL/dW|| / ||W|| per layer:")
for lyr_idx in range(model_manual.n_layers):
    gw_norm = float(np.linalg.norm(model_manual.grads_W[lyr_idx]))
    w_norm  = float(np.linalg.norm(model_manual.weights[lyr_idx]))
    ratio   = gw_norm / (w_norm + 1e-12)
    print(f"  Layer {lyr_idx + 1}: ||grad||={gw_norm:.6f}  ||W||={w_norm:.6f}  "
          f"ratio={ratio:.6f}")
print("\nA healthy ratio is typically 0.01-0.1. Much smaller -> gradient vanishing.")
print("Much larger -> gradient explosion, may require gradient clipping.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Chain rule is the engine of backprop:** Every backward function computes one
   application of the chain rule. The total backward pass chains these together from
   output to input — exactly the same mathematics as differentiating by hand, just
   automated layer-by-layer.

2. **Caching is essential:** The forward pass must store intermediate activations
   ($\mathbf{Z}_l$, $\mathbf{A}_l$) that are needed for the backward computation.
   This is the purpose of the `cache` dictionaries.

3. **Gradient checking validates correctness:** Any backprop bug will produce a
   relative error $> 10^{-3}$ against the finite-difference estimate. Run gradient
   checks whenever you implement a new layer type.

4. **Gradient norms decrease with depth:** Even with ReLU activations, the gradient
   at the first layer is typically smaller than at the last. This deepens with the
   number of layers — motivating skip connections (ResNets) and careful initialisation
   (see **05-08**).

5. **PyTorch autograd does the same thing:** Our `ManualMLP` and `PyTorchMLP` converge
   to similar test MSE. Autograd is a general-purpose system for the same chain-rule
   computation we coded by hand — but it handles arbitrary computational graphs
   automatically. The internals are explored in **05-07**.

---

### What's Next

- **05-07 — PyTorch Autograd:** Inspect `grad_fn` chains, build custom autograd
  `Function` classes, understand leaf vs. non-leaf tensors.
- **05-08 — Weight Initialisation:** Xavier, He/Kaiming, orthogonal — how initialisation
  controls gradient flow from the start of training.
